In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import ast
import pickle

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
import os
print(os.getcwd())

C:\Users\dell\Desktop\Movie\notebook


In [4]:
movies = pd.read_csv("../data/tmdb_5000_movies.csv")
credits = pd.read_csv("../data/tmdb_5000_credits.csv")

credits = credits.rename(columns={"movie_id": "id"})

movies = movies.merge(
    credits,
    on=["id", "title"]
)

In [5]:
movies.duplicated().sum()

np.int64(0)

In [6]:
# Movies with missing overview
movies[movies["overview"].isnull()][["title", "overview", "tagline"]]

,title,overview,tagline
2656,Chiamatemi Francesco - Il Papa della gente,NaN,NaN
4140,"To Be Frank, Sinatra at 100",NaN,NaN
4431,Food Chains,NaN,NaN


In [7]:
movies = movies.dropna(subset=["overview"])

In [8]:
movies["tagline"] = movies["tagline"].fillna("")

In [9]:
# Drop the unnecessary column
movies = movies.drop(columns=["homepage","original_language","budget","production_countries","release_date","status","runtime","revenue","spoken_languages","original_title"])

In [10]:
# To see how many movies are empty in genres, keywords, cast, crew
(
    (movies["genres"] == "[]") &
    (movies["keywords"] == "[]") &
    (movies["cast"] =="[]") &
    (movies["crew"]=="[]")
).sum()

np.int64(13)

In [11]:
# 1. Identify rows where ALL text metadata lists are empty strings "[]"
empty_movies_mask = (
    (movies["genres"] == "[]") &
    (movies["keywords"] == "[]") &
    (movies["cast"] == "[]") &
    (movies["crew"] == "[]")
)

# Filter the DataFrame and display the movie titles
movie_names = movies[empty_movies_mask]["title"]
print(movie_names)

# 2. Droping them so they don't break recommendation logic later
movies = movies[~empty_movies_mask]
print(f"Cleaned dataset shape: {movies.shape}")

4068                                            Sharkskin
4118                               Hum To Mohabbat Karega
4314                                            Crowsnest
4458                                  Harrison Montgomery
4504                              Light from the Darkroom
4553                           America Is Still the Place
4562                             The Little Ponderosa Zoo
4566                                         Diamond Ruff
4571    Rise of the Entrepreneur: The Search for a Bet...
4633                                          Death Calls
4657                                            UnDivided
4674                     Short Cut to Nirvana: Kumbh Mela
4716    The Blood of My Brother: A Story of Death in Iraq
Name: title, dtype: object
Cleaned dataset shape: (4787, 12)


In [12]:
# To see how many are ""
print(f"vote_average: {(movies['vote_average'] == "").sum()}")
print(f"vote_count: {(movies['vote_count'] == "[]").sum()}")
print(f"genres: {(movies["genres"] == "[]").sum()}")
print(f"keywords: {(movies["keywords"] == "[]").sum()}")
print(f"tagline: {(movies["tagline"] == "").sum()}")
print(f"overview: {(movies["overview"] == "").sum()}")
print(f"cast: {(movies["cast"]=="[]").sum()}")
print(f"crew: {(movies["crew"]=="[]").sum()}")

vote_average: 0
vote_count: 0
genres: 15
keywords: 398
tagline: 828
overview: 0
cast: 29
crew: 15


In [13]:
def convert(obj):
    L=[]
    for i in ast.literal_eval(obj):
        L.append(i['name'])
    return L
movies['genres'] = movies['genres'].apply(convert)
movies['keywords']=movies['keywords'].apply(convert)

In [14]:
print(movies['keywords'][0])

['culture clash', 'future', 'space war', 'space colony', 'society', 'space travel', 'futuristic', 'romance', 'space', 'alien', 'tribe', 'alien planet', 'cgi', 'marine', 'soldier', 'battle', 'love affair', 'anti war', 'power relations', 'mind and soul', '3d']


In [15]:
# Top 3 actor
def convert3(obj):
    L=[]
    counter = 0
    for i in ast.literal_eval(obj):
        if counter != 3:
            L.append(i['name'])
            counter+=1
        else:
            break
    return L
movies['cast'] = movies['cast'].apply(convert3)
print(movies['cast'][0])

['Sam Worthington', 'Zoe Saldana', 'Sigourney Weaver']


In [16]:
# Director Name
def fetch_director(obj):
    L=[]
    for i in ast.literal_eval(obj):
        if i['job']=='Director':
            L.append(i['name'])
            break
    return L
movies['crew']=movies['crew'].apply(fetch_director)
print(movies['crew'][0])

['James Cameron']


In [17]:
movies['overview'] = movies['overview'].apply(lambda x: x.split())
print(movies['overview'][0])

['In', 'the', '22nd', 'century,', 'a', 'paraplegic', 'Marine', 'is', 'dispatched', 'to', 'the', 'moon', 'Pandora', 'on', 'a', 'unique', 'mission,', 'but', 'becomes', 'torn', 'between', 'following', 'orders', 'and', 'protecting', 'an', 'alien', 'civilization.']


In [18]:
# 1. Create a function to strip spaces from list elements
def collapse_spaces(L):
    return [i.replace(" ", "") for i in L]

# 2. Apply to your processed columns cleanly
movies['genres'] = movies['genres'].apply(collapse_spaces)
movies['keywords'] = movies['keywords'].apply(collapse_spaces)
movies['cast'] = movies['cast'].apply(collapse_spaces)
movies['crew'] = movies['crew'].apply(collapse_spaces)
print(movies['crew'][0])

['JamesCameron']


In [19]:
movies["tags"] = (
    movies["overview"]
    + movies["genres"] * 2
    + movies["keywords"]
    + movies["cast"] 
    + movies["crew"] * 2
)

movies["tags"] = movies["tags"].apply(
    lambda x: " ".join(x).lower()
)

In [20]:
new_df = movies[["id", "title", "tags"]].copy()

In [21]:
tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words="english"
)

tfidf_vectors = tfidf.fit_transform(new_df["tags"])

print(tfidf_vectors.shape)

(4787, 5000)


In [22]:
tfidf_similarity = cosine_similarity(tfidf_vectors)

print(tfidf_similarity.shape)

(4787, 4787)


In [23]:
def recommend_tfidf(movie):
    if movie not in new_df["title"].values:
        return []

    movie_index = new_df[
        new_df["title"] == movie
    ].index[0]

    distances = tfidf_similarity[movie_index]

    movie_list = sorted(
        list(enumerate(distances)),
        key=lambda x: x[1],
        reverse=True
    )[1:6]

    recommendations = []

    for index, score in movie_list:
        recommendations.append({
            "title": new_df.iloc[index]["title"],
            "similarity_score": float(score)
        })

    return recommendations

In [24]:
recommend_tfidf("Batman Begins")

[{'title': 'The Dark Knight', 'similarity_score': 0.4178386874155254},
 {'title': 'The Dark Knight Rises', 'similarity_score': 0.4018508778819877},
 {'title': 'The Prestige', 'similarity_score': 0.27485771091252376},
 {'title': 'Batman Returns', 'similarity_score': 0.25558297271774794},
 {'title': 'Batman v Superman: Dawn of Justice',
  'similarity_score': 0.25365349663152265}]

In [25]:
recommend_tfidf("Toy Story")

[{'title': 'Toy Story 2', 'similarity_score': 0.5647230626830464},
 {'title': 'Toy Story 3', 'similarity_score': 0.5081412599815971},
 {'title': 'The 40 Year Old Virgin', 'similarity_score': 0.30550336017386387},
 {'title': 'Cars 2', 'similarity_score': 0.215012895234944},
 {'title': 'Cars', 'similarity_score': 0.1908132001021946}]

In [26]:
recommend_tfidf("Cars 2")

[{'title': 'Cars', 'similarity_score': 0.5237780595713297},
 {'title': 'The Adventures of Rocky & Bullwinkle',
  'similarity_score': 0.24113072368625754},
 {'title': 'Ice Age: Continental Drift',
  'similarity_score': 0.2302811694963896},
 {'title': 'Herbie Fully Loaded', 'similarity_score': 0.22323545170023396},
 {'title': 'Bolt', 'similarity_score': 0.21857707596769843}]

In [27]:
recommend_tfidf("The Host")

[{'title': 'Gattaca', 'similarity_score': 0.26751346474564397},
 {'title': 'In Time', 'similarity_score': 0.25640134778003043},
 {'title': 'Lord of War', 'similarity_score': 0.24054706469937304},
 {'title': 'The 5th Wave', 'similarity_score': 0.22088176798192216},
 {'title': 'Titan A.E.', 'similarity_score': 0.2153185114622459}]

In [61]:
with open("../model/tfidf_vectorizer_recommend.pkl", "wb") as file:
    pickle.dump(tfidf, file)

print("TF-IDF vectorizer saved successfully!")

TF-IDF vectorizer saved successfully!


In [62]:
with open("../model/tfidf_similarity_recommend.pkl", "wb") as file:
    pickle.dump(tfidf_similarity, file)

print("TF-IDF similarity matrix saved successfully!")

TF-IDF similarity matrix saved successfully!


In [63]:
new_df.to_pickle("../model/movies_recommendation.pkl")

print("Movie recommendation data saved successfully!")

Movie recommendation data saved successfully!


In [64]:
import pickle
import pandas as pd

# Load TF-IDF vectorizer
with open("../model/tfidf_vectorizer_recommend.pkl", "rb") as file:
    recommend_vectorizer = pickle.load(file)

# Load TF-IDF similarity matrix
with open("../model/tfidf_similarity_recommend.pkl", "rb") as file:
    recommend_similarity = pickle.load(file)

# Load movie data
recommend_movies = pd.read_pickle("../model/movies_recommendation.pkl")

print("Recommendation artifacts loaded successfully!")

Recommendation artifacts loaded successfully!


In [65]:
print("TF-IDF vocabulary:", len(recommend_vectorizer.vocabulary_))
print("Similarity matrix:", recommend_similarity.shape)
print("Movies dataframe:", recommend_movies.shape)

TF-IDF vocabulary: 5000
Similarity matrix: (4787, 4787)
Movies dataframe: (4787, 3)


In [66]:
def test_loaded_recommender(movie):

    if movie not in recommend_movies["title"].values:
        return []

    movie_index = recommend_movies[
        recommend_movies["title"] == movie
    ].index[0]

    distances = recommend_similarity[movie_index]

    movie_list = sorted(
        list(enumerate(distances)),
        key=lambda x: x[1],
        reverse=True
    )[1:6]

    recommendations = []

    for index, score in movie_list:
        recommendations.append({
            "title": recommend_movies.iloc[index]["title"],
            "similarity_score": float(score)
        })

    return recommendations

In [67]:
test_loaded_recommender("Batman Begins")

[{'title': 'The Dark Knight', 'similarity_score': 0.4178386874155254},
 {'title': 'The Dark Knight Rises', 'similarity_score': 0.4018508778819877},
 {'title': 'The Prestige', 'similarity_score': 0.27485771091252376},
 {'title': 'Batman Returns', 'similarity_score': 0.25558297271774794},
 {'title': 'Batman v Superman: Dawn of Justice',
  'similarity_score': 0.25365349663152265}]

In [28]:
new_df.to_pickle("../model/movies_recommendation.pkl")

In [29]:
print(new_df.shape)
print(new_df.columns.tolist())
print(new_df.head())

(4787, 3)
['id', 'title', 'tags']
       id                                     title  \
0   19995                                    Avatar   
1     285  Pirates of the Caribbean: At World's End   
2  206647                                   Spectre   
3   49026                     The Dark Knight Rises   
4   49529                               John Carter   

                                                tags  
0  in the 22nd century, a paraplegic marine is di...  
1  captain barbossa, long believed to be dead, ha...  
2  a cryptic message from bond’s past sends him o...  
3  following the death of district attorney harve...  
4  john carter is a war-weary, former military ca...  


In [30]:
print(tfidf_similarity.shape)

(4787, 4787)
